<a href="https://colab.research.google.com/github/obieshka/Python-2025-/blob/hw_7/%D0%BF%D1%80%D0%B0%D0%BA7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
from tqdm import tqdm

BASE_URL = "https://lifehacker.ru/topics/technology/"

def get_page_html(url):
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        return response.text
    except requests.RequestException as e:
        print(f"Ошибка при загрузке {url}: {e}")
        return None

def get_article_links(page_html):
    soup = BeautifulSoup(page_html, 'lxml')
    links = []
    cards = soup.find_all('div', class_='article-card__small-wrapper')
    for card in cards:
        link_tag = card.find('a', class_='lh-small-article-card__link')
        if link_tag and link_tag.get('href'):
            href = link_tag['href']
            if href.startswith('/'):
                href = "https://lifehacker.ru" + href
            links.append(href)
    return links

def parse_article(article_url):
    html = get_page_html(article_url)
    if not html:
        return None, None
    soup = BeautifulSoup(html, 'lxml')
    title_tag = soup.find('h1', class_='article-card__title')
    content_div = soup.find('article', class_='single-article__post-content single-article__content-container')
    title = title_tag.get_text(strip=True) if title_tag else None
    if content_div:
        content = ' '.join(content_div.stripped_strings)
    else:
        content = None
    return title, content

def main():
    articles_data = []
    total_pages = 10
    print("Сбор ссылок на статьи с первых 10 страниц рубрики 'Технологии'...")
    all_links = []
    for page_num in tqdm(range(1, total_pages + 1), desc="Обработка страниц списка"):
        url = f"{BASE_URL}?page={page_num}"
        html = get_page_html(url)
        if html:
            links = get_article_links(html)
            all_links.extend(links)
        else:
            print(f"Не удалось загрузить страницу {page_num}")
        time.sleep(0.5)
    print(f"Найдено {len(all_links)} ссылок на статьи.")
    print("Парсинг статей...")
    for link in tqdm(all_links, desc="Парсинг статей"):
        title, content = parse_article(link)
        if title and content:
            articles_data.append({
                'url': link,
                'title': title,
                'content': content
            })
        else:
            print(f"Не удалось извлечь данные из {link}")
        time.sleep(0.1)
    df = pd.DataFrame(articles_data)
    print(f"Собрано {len(df)} статей.")
    df.to_csv('lifehacker_technology_articles.csv', index=False, encoding='utf-8-sig')
    print("Данные сохранены в lifehacker_technology_articles.csv")
    return df

if __name__ == "__main__":
    df = main()
    print(df.head())

Сбор ссылок на статьи с первых 10 страниц рубрики 'Технологии'...


Обработка страниц списка: 100%|██████████| 10/10 [00:12<00:00,  1.28s/it]


Найдено 300 ссылок на статьи.
Парсинг статей...


Парсинг статей:   0%|          | 1/300 [00:01<05:03,  1.01s/it]

DEBUG: HTML сохранён в debug_article.html


Парсинг статей:  59%|█████▉    | 177/300 [03:34<08:32,  4.16s/it]

Ошибка при загрузке https://lifehacker.ru/kak-podgotovit-smartfon-k-otpusku/: HTTPSConnectionPool(host='lifehacker.ru', port=443): Read timed out. (read timeout=10)
Не удалось извлечь данные из https://lifehacker.ru/kak-podgotovit-smartfon-k-otpusku/


Парсинг статей: 100%|██████████| 300/300 [06:39<00:00,  1.33s/it]

Собрано 299 статей.
Данные сохранены в lifehacker_technology_articles.csv
                                                 url  \
0  https://lifehacker.ru/honor-600-ili-honor-600-...   
1  https://lifehacker.ru/v-akkaunt-google-teper-m...   
2  https://lifehacker.ru/apple-vypustit-11-novyx-...   
3  https://lifehacker.ru/gosuslugi-mogut-prevrati...   
4  https://lifehacker.ru/chatgpt-ustroil-xakersku...   

                                               title  \
0            Что лучше — Honor 600 или Honor 600 Pro   
1  В аккаунт Google теперь можно входить с помощь...   
2  Apple выпустит 11 новых компьютеров Mac за 2 г...   
3  «Госуслуги» могут превратиться в соцсеть с чат...   
4  Вышедшие из-под контроля боты OpenAI «сбежали»...   

                                             content  
0  Номерная линейка смартфонов Honor год от года ...  
1  Google добавила новый способ подтвердить лично...  
2  Apple готовит крупнейшее обновление линейки Ma...  
3  Власти рассматривают масштабн